> 在线讲义：[第 27 章 · MoE：很多个 FFN 专家，每个词只走其中几个](https://wikieden.github.io/beyond-transformer/lessons/27-moe-router.html) · 本书《超越 Transformer》

# 第 27 章 · MoE：很多个 FFN 专家，每个词只走其中几个

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/27-moe-router.html) · 仓库里 `beyond-transformer/notes/lessons/27-moe-router.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 router | top-2 softmax | `topk_gate`、`moe_forward` | 第 1 个 token → 专家 0、2 |
| 第三步 噪声 | 式 (4) | `noisy_logits(x, w_noise, noise)` | 选择随噪声变 |
| 第四步 参数账 | 总 / 激活 | `param_count(...)` | 46.7B / 12.9B |

输出是加权和；noisy top-k：打分加上可学幅度的噪声；总参数和激活参数的账。
完整的 MoE FFN（top-k router + 三种均衡方法）是里程碑 T-M6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
E, D = 4, 3
WG = np.array([[1.0, -0.5, 0.2, 0.0], [0.3, 1.0, -0.4, 0.2], [-0.2, 0.1, 1.0, 0.5]])   # D × E 的 router
TOKENS = np.array([[1.0, 0.2, 0.0], [0.0, 1.0, 0.3], [0.1, 0.0, 1.0], [0.5, 0.5, 0.5]])
EXPERT_SCALE = np.array([1.0, 2.0, -1.0, 0.5])   # 玩具专家：专家 e 的输出 = scale_e · x

**函数 `topk_gate`**：Mixtral 式（式 1）：Softmax(TopK(x·W_g))：只保留前 k 个分数，其余设成 −∞ 再 softmax。

In [ ]:
def topk_gate(logits, k=2):
    """Mixtral 式（式 1）：Softmax(TopK(x·W_g))：只保留前 k 个分数，其余设成 −∞ 再 softmax。"""
    idx = np.argsort(-logits)[:k]
    g = np.zeros_like(logits)
    z = np.exp(logits[idx] - logits[idx].max())
    g[idx] = z / z.sum()
    return idx, g

**函数 `moe_forward`**：小工具。

In [ ]:
def moe_forward(x, k=2):
    logits = x @ WG
    idx, g = topk_gate(logits, k)
    y = sum(g[e] * EXPERT_SCALE[e] * x for e in idx)
    return logits, idx, g, y

**函数 `noisy_logits`**：稀疏门控 MoE 式 (4)：H(x) = x·W_g + 噪声 · Softplus(x·W_noise)。

In [ ]:
def noisy_logits(x, w_noise, noise):
    """稀疏门控 MoE 式 (4)：H(x) = x·W_g + 噪声 · Softplus(x·W_noise)。"""
    return x @ WG + noise * np.log1p(np.exp(x @ w_noise))

**函数 `param_count`**：SwiGLU 专家每个 3·d·d_ff。返回 (总参数, 每个 token 激活的参数)。

In [ ]:
def param_count(d, d_ff, n_exp, k, layers, vocab, attn_per_layer, shared=0):
    """SwiGLU 专家每个 3·d·d_ff。返回 (总参数, 每个 token 激活的参数)。"""
    expert = 3 * d * d_ff
    router = d * n_exp
    per_layer_total = attn_per_layer + router + n_exp * expert + shared * expert
    per_layer_active = attn_per_layer + router + (k + shared) * expert
    emb = 2 * vocab * d
    return layers * per_layer_total + emb, layers * per_layer_active + emb

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 router 打分、选 top-2、在选中的专家上 softmax")
for t in TOKENS:
    lg, idx, g, y = moe_forward(t)
    print(f"  token {t.tolist()}：分数 {np.round(lg, 2).tolist()} → 专家 {sorted(idx.tolist())}，权重 {np.round(g, 3).tolist()}，输出 {np.round(y, 3).tolist()}")
print("\n第二步 noisy top-k")
t = TOKENS[3]
wn = np.full((D, E), 0.3)
for noise in (np.zeros(E), np.array([0.0, 0.6, -0.5, 0.3]), np.array([-1.0, 0.0, 0.0, 0.5])):
    h = noisy_logits(t, wn, noise)
    idx, _ = topk_gate(h)
    print(f"  噪声 {noise.tolist()}：分数 {np.round(h, 2).tolist()} → 选专家 {sorted(idx.tolist())}")

In [ ]:
print("\n第三步 总参数 vs 激活参数（Mixtral 8×7B 的配置：d = 4096、d_ff = 14336、8 个专家选 2、32 层）")
attn = 4096 * 4096 * 2 + 4096 * 1024 * 2    # Q、O 是 d × d；K、V 是 d × (8 头 × 128)
tot, act = param_count(4096, 14336, 8, 2, 32, 32000, attn)
print(f"  总参数 {tot / 1e9:.1f}B，每个 token 激活 {act / 1e9:.1f}B（论文：47B / 13B）")

## 3. 代码里才看得到的

- **两种权重写法**：Mixtral 先取 top-k 再 softmax；Switch、DeepSeek 等是先对全部专家 softmax 再取 top-k（权重不再加起来是 1，有的会再归一化）。两种在 k = 1 时差别很大，读代码要看清楚是哪种。
- **专家输出是“逐 token 加权”**：同一个专家对不同 token 的权重不同；这里的玩具专家是线性的，真实专家是 SwiGLU。

## 4. 自己改一改

- 把 `topk_gate` 的 k 改成 1，第 4 个 token 的输出变成多少？
- 把 `param_count` 的专家数改成 16、k 仍为 2，总参数和激活参数各变多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 医院分科：知识多（总参数），看病快（激活参数）；只有 FFN 分科，注意力共用。
> 打分、取前 k、只在选中里 softmax，加权求和。
> 加点噪声防富者愈富，推理时不加。
> 八乘七 B 不是五十六：共用的不重复，四十七 B 装着、十三 B 在算；省计算不省显存。